# TP4 · Detección de personas y vehículos desde drones
**Integrantes:** [completar] · **Fecha:** 8 de octubre de 2026  
**Dataset:** VisDrone2019-DET · **Modelo:** YOLO11n · **Pregunta:** ¿qué tan bien detecta un modelo pequeño objetos aéreos, densos y pequeños?

**PRESENTACIÓN:** recorré las celdas marcadas así, o abrí `presentacion.html` (8–10 min).  
**APRENDIZAJE:** ampliación conceptual. **EJECUCIÓN:** instalación, descarga o cómputo.

Índice: 1 Portada · 2 Problema · 3 Objetivos · 4 Fundamentos · 5 Entorno · 6 Datos · 7 Exploración · 8 Protocolo · 9 Entrenamiento · 10 Evaluación · 11 Errores · 12 Conclusiones · 13 Reproducción.

## 2. Problema y robótica · PRESENTACIÓN
Un dron ve personas y vehículos desde alturas y ángulos variables. Los objetos pueden ocupar pocos píxeles y solaparse. La detección devuelve clase, caja y confianza; es una entrada para seguimiento, conteo y decisiones de navegación, pero no reemplaza por sí sola una política de control.

## 3. Objetivos y pregunta experimental · PRESENTACIÓN
**Objetivo:** ajustar YOLO11n preentrenado a las diez clases de VisDrone. **Pregunta:** ¿cuáles clases y escenas detecta mejor, y dónde falla? El smoke test sólo verifica que el flujo funciona; el perfil full es el experimento académico. Una comparación 640/960 puede hacerse después cambiando sólo resolución y batch registrado.

## 4. Fundamentos · APRENDIZAJE
Una *bounding box* contiene un objeto. En formato YOLO se expresa como `clase x_centro y_centro ancho alto`, con coordenadas divididas por ancho/alto de imagen. YOLO predice cajas y clases en una sola pasada. *Transfer learning* inicia con pesos aprendidos en otro conjunto y reajusta el modelo a VisDrone; las clases COCO de los pesos iniciales no son equivalentes a las diez clases finales.

**Precision**: proporción de predicciones correctas. **Recall**: proporción de objetos reales encontrados. **mAP@0.5** promedia precisión cuando IoU ≥ 0.5; **mAP@0.5:0.95** promedia varios umbrales y exige cajas mejor ubicadas.

## 5. Entorno local y Colab · EJECUCIÓN
**Qué hace:** localiza el proyecto y consulta versiones/GPU. **Por qué:** el entrenamiento requiere CUDA y Colab puede no ofrecer GPU gratis. **Cómo leerlo:** `cuda=True` y el nombre de GPU confirman que PyTorch puede usarla; el número CUDA de `nvidia-smi` representa soporte del driver, no un toolkit instalado. No entrenar en CPU por accidente.

En Windows usá el kernel `TP4 VisDrone YOLO`; en Colab cloná el repositorio y configurá `TP4_ROOT` al directorio clonado. El almacenamiento de Colab se pierde al cerrar la sesión; respaldá `runs/` en Drive si lo necesitás.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('TP4_ROOT', Path.cwd())).resolve()
if not (ROOT / 'config.yaml').exists() and (ROOT.parent / 'config.yaml').exists(): ROOT = ROOT.parent
if not (ROOT / 'config.yaml').exists(): raise FileNotFoundError('Definí TP4_ROOT al directorio tp4-yolo')
sys.path.insert(0, str(ROOT))
from tp4.experiment import versions, resolve, latest_run
from tp4.report import presentation
import torch
print(json.dumps(versions(), indent=2, ensure_ascii=False))
print('CUDA disponible:', torch.cuda.is_available())

## 6. Dataset y conversión · APRENDIZAJE
VisDrone2019-DET conserva particiones oficiales train (6471), val (548), test-dev (1610). Las categorías originales 1–10 se convierten a 0–9: pedestrian, people, bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor. *Pedestrian* es una persona de pie o caminando; *people* incluye otras posturas.

Ejemplo: imagen 1000×500, caja original `(x=100,y=50,w=200,h=100)`, categoría 4 (car), score 1. Centro `(200,100)`; salida YOLO `3 0.2 0.2 0.2 0.2`. Score 0 marca región ignorada. Se excluyen categorías fuera de 1–10 y se registran cajas recortadas al borde. Quitar regiones ignoradas de las etiquetas **no** equivale necesariamente al tratamiento del evaluador oficial.

### Descarga y auditoría · EJECUCIÓN
**Qué hace:** baja tres ZIP documentados, conserva originales, verifica ZIP e imágenes y convierte etiquetas. **Por qué:** evita descargas repetidas y permite auditar la conversión. **Salida:** `data/audit.json` muestra cantidades reales; comparalas con las esperadas, sin ocultar diferencias. La primera ejecución puede tardar varios minutos.

In [ ]:
RUN_PREPARE = os.environ.get('TP4_PREPARE', '0') == '1'
if RUN_PREPARE:
    from tp4.data import prepare_all
    audit = prepare_all(ROOT / 'data')
    print({k: (v['images'], v['rows']) for k, v in audit.items()})
else:
    print('Preparación omitida. Ejecutá python -m tp4.cli prepare o TP4_PREPARE=1.')

## 7. Exploración de datos · PRESENTACIÓN
**Qué hace:** muestra distribución de clases, áreas relativas y cajas convertidas. **Por qué:** el desbalance y los objetos pequeños condicionan el aprendizaje. **Cómo leerlo:** el histograma de áreas usa `área_caja / área_imagen`; no mide desempeño por tamaño. La densidad de objetos se registra en la auditoría.

In [ ]:
from IPython.display import display, Image, Markdown
for name in ('audit.png', 'box_review.png'):
    image = ROOT / 'data' / name
    if image.exists(): display(Image(filename=str(image)))
    else: display(Markdown(f'Pendiente: {name}. Ejecutá preparación para generarlo.'))
audit_file = ROOT / 'data' / 'audit.json'
if audit_file.exists():
    audit = json.loads(audit_file.read_text(encoding='utf-8'))
    display({k: {'imágenes': v['images'], 'objetos/imagen': round(v['mean_objects_per_image'], 1), 'filas': v['rows']} for k,v in audit.items()})

## 8. Particiones y protocolo · APRENDIZAJE
Train ajusta pesos; val selecciona checkpoint y parámetros; test-dev etiquetado queda reservado para evaluación final. No redistribuir imágenes al azar entre particiones: hacerlo contamina la comparación. El smoke usa 128 imágenes train y 32 val escogidas con semilla 42, sin tocar test-dev. Los resultados de val no se presentan como test independiente.

## 9. Entrenamiento · EJECUCIÓN
**Qué hace:** ajusta YOLO11n con pesos preentrenados y guarda `best.pt` y `last.pt`. **Por qué:** se adapta la representación previa a clases aéreas. **Cómo leerlo:** pérdidas decrecientes y métricas val orientan el diagnóstico; dos épocas smoke no permiten concluir calidad final. Activá `TP4_PROFILE=smoke` y `TP4_TRAIN=1` para prueba, o `full` para el entrenamiento académico. Se exige GPU CUDA. Si hay OOM, bajá batch explícitamente en `config.yaml` o por CLI y registrá el cambio. En Windows `workers=0` evita problemas de procesos.

In [ ]:
PROFILE = os.environ.get('TP4_PROFILE', 'presentation')
RUN_TRAIN = os.environ.get('TP4_TRAIN', '0') == '1'
if RUN_TRAIN:
    from ultralytics import YOLO
    import ultralytics.utils as ultralytics_utils
    from tp4.experiment import device_or_raise, start_run, finish_run, metrics_dict
    import time
    if PROFILE not in ('smoke', 'full'): raise ValueError('Para entrenar elegí smoke o full')
    settings = resolve(ROOT, PROFILE)
    ultralytics_utils.WEIGHTS_DIR = ROOT / 'weights'
    ultralytics_utils.WEIGHTS_DIR.mkdir(exist_ok=True)
    device = device_or_raise(settings['device'])
    run = start_run(ROOT, PROFILE, settings)
    began = time.perf_counter()
    try:
        model = YOLO(settings['model'])  # pesos COCO preentrenados; la cabeza se adapta a 10 clases
        model.train(data=settings['data_yaml'], epochs=settings['epochs'], imgsz=settings['imgsz'], batch=settings['batch'], workers=settings['workers'], seed=settings['seed'], device=device, project=str(run), name='train', exist_ok=True, plots=True)
        best = run / 'train' / 'weights' / 'best.pt'
        val = YOLO(str(best)).val(data=settings['data_yaml'], split='val', imgsz=settings['imgsz'], batch=settings['batch'], workers=settings['workers'], device=device, project=str(run), name='validation', exist_ok=True, plots=True)
        finish_run(run, elapsed=time.perf_counter()-began, metrics=metrics_dict(val))
    except Exception as exc:
        finish_run(run, elapsed=time.perf_counter()-began, error=exc)
        raise
    print('Artefactos:', run)
else:
    print('Entrenamiento omitido. Se cargarán artefactos guardados.')

## 10. Evaluación · PRESENTACIÓN
**Qué hace:** lee un run completo si existe; de lo contrario, uno smoke. **Por qué:** la exposición no depende de las celdas de descarga y entrenamiento. **Cómo leerlo:** la etiqueta SMOKE marca resultados preliminares. `best.pt` se evalúa en val; para test-dev final usá `python -m tp4.cli test RUTA_RUN`. Las métricas usan el evaluador estándar de Ultralytics y no son resultados oficiales del benchmark.

In [ ]:
run = latest_run(ROOT, 'full') or latest_run(ROOT, 'smoke')
if run:
    info = json.loads((run/'run.json').read_text(encoding='utf-8'))
    print('PERFIL:', info['settings']['profile'].upper(), '| run:', run)
    display(info.get('metrics', {}))
    for name in ('results.png', 'confusion_matrix.png', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg'):
        candidate = run / ('train' if name == 'results.png' else 'validation') / name
        if candidate.exists(): display(Image(filename=str(candidate)))
else:
    display(Markdown('**Sin resultados guardados.** Prepará los datos y ejecutá smoke o full.'))

## 11. Análisis de errores y límites · APRENDIZAJE
Compará las predicciones con las cajas reales para identificar falsos positivos (caja sin objeto) y falsos negativos (objeto sin caja). Revisá oclusión, multitudes y objetos de pocos píxeles. La matriz de confusión muestra confusiones entre clases; una caja mal ubicada también reduce mAP. No deduzcas precisión por tamaño a partir de unas imágenes: requiere definir rangos de área y evaluar todas las instancias. Para latencia, medí warm-up y sincronizá CUDA; registrá GPU, resolución, batch y número de repeticiones.

In [ ]:
if run:
    from ultralytics import YOLO
    import time
    if os.environ.get('TP4_LATENCY', '0') == '1':
        sample = next((ROOT/'data'/'yolo'/'images'/'val').glob('*.jpg'))
        detector = YOLO(str(run/'train'/'weights'/'best.pt'))
        dev = '0' if torch.cuda.is_available() else 'cpu'
        for _ in range(5): detector.predict(str(sample), imgsz=640, device=dev, verbose=False)
        if dev == '0': torch.cuda.synchronize()
        begin = time.perf_counter()
        for _ in range(20): detector.predict(str(sample), imgsz=640, device=dev, verbose=False)
        if dev == '0': torch.cuda.synchronize()
        print({'ms_por_imagen': (time.perf_counter()-begin)*1000/20, 'dispositivo': dev, 'imgsz': 640, 'batch': 1, 'warmup': 5, 'repeticiones': 20})
    else: print('Latencia pendiente: TP4_LATENCY=1 para medir 5 warm-up + 20 inferencias.')

In [ ]:
if run:
    gallery = run / 'error_gallery.jpg'
    if gallery.exists():
        display(Image(filename=str(gallery)))
        display(json.loads((run/'error_sample.json').read_text(encoding='utf-8')))
    else: print('Galería de errores pendiente: python -m tp4.cli analyze RUTA_RUN')
    latency = run / 'latency.json'
    if latency.exists(): display(json.loads(latency.read_text(encoding='utf-8')))

## 12. Conclusiones · PRESENTACIÓN
Si sólo existe un run smoke: **se verificó el pipeline, aún no hay evidencia suficiente sobre la calidad del detector**. Si existe full: redactar aquí conclusiones basadas en métricas por clase, matriz y errores visuales antes de entregar. Separar siempre validación del resultado final de test-dev. La comparación con un COCO sin adaptar sería inválida porque las clases no coinciden.

## 13. Referencias y reproducción · PRESENTACIÓN
- [VisDrone-Dataset, fuente y condiciones](https://github.com/VisDrone/VisDrone-Dataset)
- [VisDrone, uso académico y derechos](https://aiskyeye.com/data-protection/)
- [VisDrone en Ultralytics: clases, splits y descarga](https://docs.ultralytics.com/datasets/detect/visdrone/)
- [YOLO11](https://docs.ultralytics.com/models/yolo11/) y [modo train](https://docs.ultralytics.com/modes/train/)
- [Instalación de PyTorch](https://pytorch.org/get-started/locally/)

Reproducir: `python -m tp4.cli prepare`, luego `python -m tp4.cli train smoke`, inspeccionar `runs/smoke/`; lanzar `python -m tp4.cli train full` sólo cuando se desee el entrenamiento largo. `python -m tp4.cli present` produce HTML autónomo. Ver README para instalación, Colab, reanudación y respaldo. Uso del dataset: consultar condiciones de sus autores; la licencia AGPL-3.0 del código Ultralytics es independiente.

In [ ]:
print('Presentación:', presentation(ROOT, run))